# Lyric Engine - Colab Production Generation

This notebook handles extracting and loading your trained genre checkpoints from Google Drive to run on a T4 GPU.

## 1. One-Time Setup
Mount your drive, clone the repository, and install all dependencies needed to run the engine.

In [11]:
import os
import sys

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the repository
if not os.path.exists('/content/lyric-engine'):
    !git clone https://github.com/SMXFREEZE/lyric-engine /content/lyric-engine

# 3. Move into the working directory
os.chdir('/content/lyric-engine')

# 4. Install dependencies
!pip install -q transformers peft accelerate bitsandbytes trl datasets \
    lyricsgenius pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer

print("Setup Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup Complete.


## 2. Checkpoint Prep
Locate your extracted folder checkpoint on Drive, or automatically unzip your uploaded `.zip` file.

In [12]:
import os
import zipfile

# Define expected Colab paths
ZIP_PATH = '/content/drive/MyDrive/genre_trap_step_2600_upload2.zip'
FOLDER_PATH = '/content/drive/MyDrive/genre_trap_step_2600'

# Local extraction directory (if zipped)
LOCAL_EXTRACT_DIR = '/content/lyric-engine/checkpoints/genre_trap'

if os.path.exists(FOLDER_PATH):
    print(f"Found existing unzipped folder: {FOLDER_PATH}")
    CKPT_DIR = FOLDER_PATH
elif os.path.exists(ZIP_PATH):
    print(f"Found zipped checkpoint at: {ZIP_PATH}")
    print(f"Extracting to: {LOCAL_EXTRACT_DIR} ...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)
    
    # Handle potentially nested folders inside the zip
    extracted_contents = os.listdir(LOCAL_EXTRACT_DIR)
    if len(extracted_contents) == 1 and os.path.isdir(os.path.join(LOCAL_EXTRACT_DIR, extracted_contents[0])):
        CKPT_DIR = os.path.join(LOCAL_EXTRACT_DIR, extracted_contents[0])
    else:
        CKPT_DIR = LOCAL_EXTRACT_DIR
    print(f"Extraction complete. Target checkpoint: {CKPT_DIR}")
else:
    raise FileNotFoundError("Could not find either the zipped checkpoint or the folder checkpoint directly on your Drive.")

Found zipped checkpoint at: /content/drive/MyDrive/genre_trap_step_2600_upload2.zip
Extracting to: /content/lyric-engine/checkpoints/genre_trap ...
Extraction complete. Target checkpoint: /content/lyric-engine/checkpoints/genre_trap


## 3. Configuration & Tokens
Setup environment variables and verify the script has access to a T4 GPU.

In [13]:
import os
import torch

# Add your HF Token if accessing gated models (e.g. Llama-3)
HF_TOKEN = '' 
if HF_TOKEN:
    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

BASE_MODEL = 'mistralai/Mistral-7B-v0.1'
GENRE = 'trap'

# Verify architecture resolved device is CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Resolved Device (Should be CUDA for T4): {device.upper()}")
assert device == 'cuda', "CUDA is not available. Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU."

Resolved Device (Should be CUDA for T4): CPU


AssertionError: CUDA is not available. Please go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.

## 4. Load Base Model and LoRA Adapter
Downloads the base model into 4-bit representation to fit comfortably in T4 VRAM, then attaches your LoRA weights.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"Loading Base Model ({BASE_MODEL}) in 4-bit mode to save VRAM...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    device_map="auto"
)

print(f"Loading Tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
except Exception:
    print("Tokenizer not found in checkpoint folder, fetching from base model directly...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f"Applying LoRA Checkpoint from: {CKPT_DIR} ...")
try:
    trained_model = PeftModel.from_pretrained(base_model, CKPT_DIR)
    print("LoRA Adapter successfully loaded and attached!")
except Exception as e:
    raise RuntimeError(f"Failed to load PEFT adapter from {CKPT_DIR}. See error: {e}")

## 5. Sanity Check Generation
Quick standalone test using `transformers` code to ensure models successfully merge without syntax errors.

In [ ]:
inputs = tokenizer("The rhythm of the night is", return_tensors="pt").to("cuda")

print("Running fast test generation...")
with torch.no_grad():
    outputs = trained_model.generate(
        **inputs, 
        max_new_tokens=20, 
        do_sample=True,
        temperature=0.7
    )

print("\n--- SANITY CHECK RESULT ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("---------------------------\n")
print("Sanity check passed!")

## 6. Full Lyric Generation Pipeline
Routes the trained model into the codebase's custom lyric engine to generate structured lyrics based on metadata rules.

In [ ]:
from src.inference.engine import LyricsEngine, SongMemory

# Initialize the Lyric Engine wrapped with your model
engine = LyricsEngine(trained_model, tokenizer, device="cuda", beam_size=4)

# Setup song memory structure to act as prompt
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f"\n======================================")
print(f"=== Generating full {GENRE.upper()} verse ===")
print(f"======================================\n")

# Run engine for the VERSE section
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for line in verse:
    print(f"  {line}")